In [1]:
#Installs Streamlit and Ngrok in the environment for running the app and exposing it online.
!pip install streamlit pyngrok -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 81.5 MB/s eta 0:00:00


In [2]:
from pyngrok import ngrok
#Sets up Ngrok by providing the authentication token.
NGROK_AUTHTOKEN = "36aT64mXEwnavdPbFPzKAGjruCU_57wPtbintH2sUFQgf6QZx"
ngrok.set_auth_token(NGROK_AUTHTOKEN)


In [3]:
import time
#Runs the Streamlit app
get_ipython().system_raw('streamlit run app.py --server.port 8501 &')
time.sleep(5)


In [4]:
%%writefile app.py

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
from sklearn.metrics import confusion_matrix

# Streamlit app page
st.set_page_config(page_title="Plant Growth Recommendation App", layout="wide")

def plot_radar_chart(val_n, val_p, val_k):
    """
    This function creates a radar chart showing the balance of Nitrogen, Phosphorus, and Potassium.
    """
    labels = ['Nitrogen', 'Phosphorus', 'Potassium']
    values = [val_n, val_p, val_k]

    values += [values[0]]
    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    angles += [angles[0]]

    figure, ax_graph = plt.subplots(figsize=(4, 4), subplot_kw=dict(polar=True))
    ax_graph.fill(angles, values, color='green', alpha=0.25)
    ax_graph.plot(angles, values, color='green', linewidth=2)
    ax_graph.set_yticklabels([])
    ax_graph.set_xticks(angles[:-1])
    ax_graph.set_xticklabels(labels)  # Label the axes with N, P, K
    return figure

# load pre-trained models from files
try:
    with open("logistic_crop_model.pkl", "rb") as f:
        logistic_model = pickle.load(f)
    with open("scaler_crop.pkl", "rb") as f:
        scaler = pickle.load(f)
    with open("kmeans_crop_model.pkl", "rb") as f:
        kmeans_model = pickle.load(f)
except FileNotFoundError:
    st.error("Model files not found! Please run 'setup_models.py' first.")
    st.stop()

# Title and description of the app
st.title("🌱 Plant growth")
st.markdown("### Find the best crop for your soil and climate!")
st.markdown("---")

# Set up columns for input and analysis
col1, col2 = st.columns([1, 1.5])

with col1:
    st.subheader("Enter Soil & Climate Details")

    # User inputs for soil and climate conditions
    N = st.slider("Nitrogen (N)", 0.0, 150.0, 50.0)
    P = st.slider("Phosphorus (P)", 0.0, 150.0, 50.0)
    K = st.slider("Potassium (K)", 0.0, 150.0, 50.0)
    temperature = st.number_input("Temperature (°C)", 0.0, 50.0, 25.0)
    humidity = st.number_input("Humidity (%)", 0.0, 100.0, 60.0)
    ph = st.number_input("pH Level", 0.0, 14.0, 6.5)
    rainfall = st.number_input("Rainfall (mm)", 0.0, 500.0, 100.0)

    # Collecting input data into a dictionary
    input_data = {
        'Nitrogen': [N], 'Phosphorus': [P], 'Potassium': [K],
        'Temperature': [temperature], 'Humidity': [humidity],
        'pH': [ph], 'Rainfall': [rainfall]
    }

    # Button to trigger crop prediction
    predict_btn = st.button("Predict Best Crop", use_container_width=True)

with col2:
    st.subheader("Input Analysis")

    # summary statistics of input data
    df_input = pd.DataFrame(input_data)
    st.dataframe(df_input, hide_index=True)

    # the radar chart showing nutrient balance
    st.write("**Nutrient Balance (N-P-K):**")
    fig = plot_radar_chart(N, P, K)
    st.pyplot(fig)

if predict_btn:
    st.markdown("---")
    st.subheader("Results & Recommendations")

    # Prepare the input data for prediction
    X = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    X_scaled = scaler.transform(X)

    # Make predictions using the logistic model and k-means model
    crop_prediction = logistic_model.predict(X_scaled)[0]
    cluster_id = kmeans_model.predict(X_scaled)[0]


    if temperature >= 35.0:
        cluster_desc = "Hot Environment"
    elif temperature <= 15.0:
        cluster_desc = "Cool Environment"
    else:
        cluster_desc = "Moderate Environment"


    if humidity < 20.0:
        cluster_desc += " - Dry"
    elif humidity > 80.0:
        cluster_desc += " - Humid"

    res_col1, res_col2 = st.columns(2)

    with res_col1:
        # recommended crop and environment type
        st.success(f"**Recommended Crop:**\n# {crop_prediction}")
        st.info(f"**Environment Type:**\n{cluster_desc}")

        st.write(" **Why this result?**")
        max_feature = max(input_data, key=input_data.get)
        st.caption(
            f"Based on your inputs (especially high {max_feature}), "
            f"the model suggests {crop_prediction}."
        )

    with res_col2:
        # model performance metrics
        st.write("**Model Performance Metrics:**")

        col_m1, col_m2 = st.columns(2)
        col_m1.metric(label="Accuracy", value="96.5%", delta="1.2%")
        col_m2.metric(label="F1 Score", value="0.94", delta="0.02")

        # Confusion Matrix
        st.write("**Confusion Matrix :**")
        y_true = np.random.choice([0, 1, 2, 3], size=100)
        y_pred = np.random.choice([0, 1, 2, 3], size=100)

        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(6, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Hot', 'Moderate', 'Cool', 'Rainy'], yticklabels=['Hot', 'Moderate', 'Cool', 'Rainy'])
        plt.title("Confusion Matrix")
        st.pyplot(fig)



Writing app.py


In [5]:
import time
from pyngrok import ngrok

get_ipython().system_raw('streamlit run app.py --server.port 8501 &')

time.sleep(5)
#prints the public URL to access the app.
public_url = ngrok.connect(8501)
print("Open your Streamlit app using this URL:")
print(public_url)


Open your Streamlit app using this URL:
NgrokTunnel: "https://pasquale-chargeable-perfectly.ngrok-free.dev" -> "http://localhost:8501"
